In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def ler_csv(nome):
    arquivo = nome if Path(nome).exists() else nome.replace('.csv', '_limpo.csv')
    return pd.read_csv(arquivo)

df1 = ler_csv('olist_orders_dataset.csv')
df2 = ler_csv('olist_order_items_dataset.csv')
df3 = ler_csv('olist_products_dataset.csv')
df4 = ler_csv('olist_order_reviews_dataset.csv')
df5 = ler_csv('olist_customers_dataset.csv')

**Limpa de Lista de Pedidos (DF1)**

In [ ]:
print(df1.info())

In [ ]:
df1.columns = ["id_pedido", "id_cliente", "status_pedido",
               "marca_temporal_compra_pedido", "pedido_aprovado_as",
               "data_entrega_pedido_transporadora",
               "data_entrega_pedido_cliente", "data_estimada_entrega_pedido"]
print(df1.head())

In [ ]:
print("\n---- Nulos por coluna ----")
print(df1.isna().sum())

print("\n---- Porcentagem de Nulos ----")
print(df1.isna().mean() * 100)

Possui nulos em três colunas específicas, analisar depois as linhas nulas.

In [ ]:
colunas_com_nulos_df1 = ['pedido_aprovado_as','data_entrega_pedido_transporadora','data_entrega_pedido_cliente']
linhas_nulas_df1 = df1[df1[colunas_com_nulos_df1].isna().any(axis = 1)]
print(linhas_nulas_df1)

In [ ]:
print("\n---- Duplicatas por colunas ----")
print(df1.duplicated().sum())

Esse dataframe não possui linhas duplicadas então é desnecessário utilizar `drop_duplicates()` nele.

Como os dataframes não possuem duplicata a única coisa a fazer para completar a limpar é padronizar os nomes das colunas e converter os tipo inapropriados de colunas.

In [ ]:
df1.columns = df1.columns.str.strip().str.lower().str.replace(' ', '_')

As colunas `marca_temporal_compra_pedido`, `pedido_aprovado_as`, `data_entrega_pedido_transporadora`, `data_entrega_pedido_cliente` e `data_estimada_entrega_pedido` estão incorretamente classificadas como `object` ao invés de `datetime`.

In [ ]:
df1['marca_temporal_compra_pedido'] = pd.to_datetime(df1['marca_temporal_compra_pedido'], errors='coerce')
df1['pedido_aprovado_as'] = pd.to_datetime(df1['pedido_aprovado_as'], errors='coerce')
df1['data_entrega_pedido_transporadora'] = pd.to_datetime(df1['data_entrega_pedido_transporadora'], errors='coerce')
df1['data_entrega_pedido_cliente'] = pd.to_datetime(df1['data_entrega_pedido_cliente'], errors='coerce')
df1['data_estimada_entrega_pedido'] = pd.to_datetime(df1['data_estimada_entrega_pedido'], errors='coerce')

compra = df1['marca_temporal_compra_pedido']
transportadora = df1['data_entrega_pedido_transporadora']
entrega = df1['data_entrega_pedido_cliente']
datas_erradas = (transportadora < compra) | (transportadora > entrega)
df1.loc[datas_erradas, 'data_entrega_pedido_transporadora'] = pd.NaT

**Limpa de Lista de Itens de Pedidos (DF2)**

In [ ]:
print(df2.info())

In [ ]:
df2.columns = ["id_pedido", "id_item_pedido", "id_produto",
               "id_vendedor", "data_limite_envio",
               "preco", "valor_frete"]
print(df2.head())

In [ ]:
print("\n---- Nulos por coluna ----")
print(df2.isna().sum())

print("\n---- Porcentagem de Nulos ----")
print(df2.isna().mean() * 100)

Dataframe sem nulos. Não é necessário limpa nesse quesito.

In [ ]:
print("\n---- Duplicatas por colunas ----")
print(df2.duplicated().sum())

Outro dataframe sem duplicatas.

A coluna `data_limite_envio` está incorretamente classificada como `object` ao invés de `datetime`.

In [ ]:
df2['data_limite_envio'] = pd.to_datetime(df2['data_limite_envio'], errors='coerce')

In [ ]:
for coluna in ['preco', 'valor_frete']:
    q1 = df2[coluna].quantile(0.25)
    q3 = df2[coluna].quantile(0.75)
    iqr = q3 - q1
    extremos = (df2[coluna] < q1 - 1.5 * iqr) | (df2[coluna] > q3 + 1.5 * iqr)
    print(coluna, 'valores atípicos:', extremos.sum())

**Limpa de Lista de Produtos (DF3)**

In [ ]:
print(df3.info())

In [ ]:
df3.columns = ["id_produto", "nome_categoria_produto",
               "comprimento_nome_produto", "comprimento_descricao_produto",
               "qtd_fotos_produto", "peso_produto_g", "comprimento_produto_cm",
               "altura_produto_cm", "largura_produto_cm"]
print(df3.head())

In [ ]:
print("\n---- Nulos por coluna ----")
print(df3.isna().sum())

print("\n---- Porcentagem de Nulos ----")
print(df3.isna().mean() * 100)

Dataframe com nulos em quase todas as colunas. Quatro colunas tem 610 valores nulos e outras quatro possuem 2 valores nulos, indicando que as mesmas linhas possuem várias colunas nulas.

In [ ]:
colunas_com_nulos_df3 = ['nome_categoria_produto','comprimento_nome_produto',
                         'comprimento_descricao_produto', 'qtd_fotos_produto',
                         'peso_produto_g','comprimento_produto_cm',
                         'altura_produto_cm', 'largura_produto_cm']
linhas_nulas_df3 = df3[df3[colunas_com_nulos_df3].isna().any(axis = 1)]
print(linhas_nulas_df3)

A hipotese anterior está correta, maior parte das linhas listadas tem múltiplas colunas nulas.

As colunas `comprimento_nome_produto`, `comprimento_descricao_produto` e `qtd_fotos_produto` estão como `float` por causa dos nulos. Usar `Int64` para manter esses valores ausentes sem excluir os produtos.

In [ ]:
df3 = df3.dropna(subset=['id_produto'])

In [ ]:
df3['comprimento_nome_produto'] = df3['comprimento_nome_produto'].astype('Int64')
df3['comprimento_descricao_produto'] = df3['comprimento_descricao_produto'].astype('Int64')
df3['qtd_fotos_produto'] = df3['qtd_fotos_produto'].astype('Int64')

In [ ]:
print("\n---- Duplicatas por colunas ----")
print(df3.duplicated().sum())

**Limpa de Lista de Reviews de Pedidos (DF4)**

In [ ]:
print(df4.info())

In [ ]:
df4.columns = ["id_avaliacao", "id_pedido", "nota_avaliacao",
               "titulo_comentario_avaliacao", "mensagem_comentario_avaliacao",
               "data_criacao_avaliacao", "marca_temporal_resposta_avaliacao"]
print(df4.head())

In [ ]:
print("\n---- Nulos por coluna ----")
print(df4.isna().sum())

print("\n---- Porcentagem de Nulos ----")
print(df4.isna().mean() * 100)

Como o briefing disse existem valores nulos nas colunas de mensagem e titulo de avaliação. Melhor opção é preencher esses nulos com 'Vazio' para indicar que o cliente não escreveu nada e eliminar a presença de nulos no dataframe.

In [ ]:
df4['titulo_comentario_avaliacao'] = df4['titulo_comentario_avaliacao'].fillna('Vazio')
df4['mensagem_comentario_avaliacao'] = df4['mensagem_comentario_avaliacao'].fillna('Vazio')

As colunas `data_criacao_avaliacao` e `marca_temporal_resposta_avaliacao` devem ser convertidas para `datetime`.

In [ ]:
df4['data_criacao_avaliacao'] = pd.to_datetime(df4['data_criacao_avaliacao'], errors='coerce')
df4['marca_temporal_resposta_avaliacao'] = pd.to_datetime(df4['marca_temporal_resposta_avaliacao'], errors='coerce')

In [ ]:
print("\n---- Duplicatas por colunas ----")
print(df4.duplicated().sum())

**Limpa de Lista de Clientes (DF5)**

In [ ]:
print(df5.info())

In [ ]:
df5.columns = ["id_cliente", "id_unica_cliente",
               "prefixo_cep_cliente", "cidade_cliente", "estado_cliente",]
print(df5.head())

In [ ]:
print("\n---- Nulos por coluna ----")
print(df5.isna().sum())

print("\n---- Porcentagem de Nulos ----")
print(df5.isna().mean() * 100)

In [ ]:
print("\n---- Duplicatas por colunas ----")
print(df5.duplicated().sum())

In [ ]:
df1.to_csv('olist_orders_dataset_limpo.csv', index=False)
df2.to_csv('olist_order_items_dataset_limpo.csv', index=False)
df3.to_csv('olist_products_dataset_limpo.csv', index=False)
df4.to_csv('olist_order_reviews_dataset_limpo.csv', index=False)
df5.to_csv('olist_customers_dataset_limpo.csv', index=False)